In [2]:
import pandas as pd
import os
import shutil
import numpy as np

# ================= 配置区域 =================
# 1. CSV 文件路径
input_csv_path = 'NIFD.csv'

# 2. 【关键修改】你的 nii 图像所在的文件夹名称
# 假设你的目录结构是：
#  - 脚本.py
#  - NIFD.csv
#  - MRI/ (这里面装着所有的 nii 文件)
source_image_dir = 'MRI' 

# 3. 输出结果的保存目录
output_root = './NIFD_Processed'

# ================= ADNI 目标列名格式 =================
target_columns = [
    'path', 'filename', 'age', 'gender', 'education', 'hispanic', 'race', 'apoe', 
    'NC', 'MCI', 'DE', 'COG', 'AD', 'mmse', 'cdr', 'cdrSum', 'Tesla', 
    'trailA', 'trailB', 'lm_imm', 'lm_del', 'boston', 'animal', 'vege', 
    'digitB', 'digitBL', 'digitF', 'digitFL', 'npiq_DEL', 'npiq_HALL', 
    'npiq_AGIT', 'npiq_DEPD', 'npiq_ANX', 'npiq_ELAT', 'npiq_APA', 'npiq_DISN', 
    'npiq_IRR', 'npiq_MOT', 'npiq_NITE', 'npiq_APP', 'faq_BILLS', 'faq_TAXES', 
    'faq_SHOPPING', 'faq_GAMES', 'faq_STOVE', 'faq_MEALPREP', 'faq_EVENTS', 
    'faq_PAYATTN', 'faq_REMDATES', 'faq_TRAVEL', 'his_CVHATT', 'his_PSYCDIS', 
    'his_Alcohol', 'his_SMOKYRS', 'his_PACKSPER', 'his_NACCFAM', 'his_CBSTROKE', 
    'his_HYPERTEN', 'his_DEPOTHR', 'gds', 'moca'
]

def process_nifd_dataset():
    # 1. 创建输出目录结构
    categories = {'NC': 0, 'MCI': 1, 'AD': 2}
    for cat in categories:
        os.makedirs(os.path.join(output_root, cat), exist_ok=True)
    
    # 2. 读取 NIFD 数据
    if not os.path.exists(input_csv_path):
        print(f"错误: 找不到CSV文件 {input_csv_path}")
        return

    print(f"正在读取 {input_csv_path}...")
    df = pd.read_csv(input_csv_path)

    # 【修复】确保 COG 列是整数，防止 1.0 != 1 的情况导致匹配失败
    # 先填充NaN为-1，然后转为整数
    df['COG'] = df['COG'].fillna(-1).astype(int)
    
    # 3. 过滤并处理每一类
    for cat_name, cog_val in categories.items():
        print(f"\n========== 正在处理类别: {cat_name} (COG={cog_val}) ==========")
        
        # 筛选对应COG值的行
        sub_df = df[df['COG'] == cog_val].copy()
        print(f"该类别在 CSV 中共有 {len(sub_df)} 条记录。")
        
        if len(sub_df) == 0:
            continue
            
        new_rows = []
        success_count = 0
        
        for idx, row in sub_df.iterrows():
            filename = str(row['filename'])
            
            # --- A. 【关键修改】构建源文件路径 ---
            # 直接去 source_image_dir (MRI) 文件夹里找，忽略 csv 里的 path
            src_path = os.path.join(source_image_dir, filename)
            
            # 检查源文件是否存在
            if not os.path.exists(src_path):
                # 尝试去掉可能存在的路径前缀（防御性编程）
                basename = os.path.basename(filename)
                src_path_retry = os.path.join(source_image_dir, basename)
                if os.path.exists(src_path_retry):
                    src_path = src_path_retry
                else:
                    # 为了不刷屏，只打印前5个错误
                    if idx < 5: 
                        print(f"❌ 文件未找到: {src_path}")
                    elif idx == 5:
                        print(f"... 更多文件未找到 (省略打印) ...")
                    continue
                
            # 构建目标路径
            dst_dir = os.path.join(output_root, cat_name)
            dst_path = os.path.join(dst_dir, filename)
            
            # 复制文件
            try:
                shutil.copy2(src_path, dst_path)
                success_count += 1
            except Exception as e:
                print(f"复制失败 {src_path}: {e}")
                continue

            # --- B. 数据对齐 ---
            new_row = {}
            
            # 1. 填充原有列
            for col in sub_df.columns:
                if col in target_columns:
                    new_row[col] = row[col]
            
            # 2. 填充缺失列
            for col in target_columns:
                if col not in new_row:
                    new_row[col] = np.nan
            
            # 3. 特殊字段修正
            # path 指向新的文件夹绝对路径
            new_row['path'] = os.path.abspath(dst_dir) + '/'
            
            # 修正标签
            new_row['NC'] = 1 if cat_name == 'NC' else 0
            new_row['MCI'] = 1 if cat_name == 'MCI' else 0
            new_row['AD'] = 1 if cat_name == 'AD' else 0
            new_row['COG'] = cog_val
            
            new_rows.append(new_row)
        
        print(f"✅ 成功复制并处理了 {success_count} 个文件。")

        # 保存该类别的 CSV
        if new_rows:
            cat_df = pd.DataFrame(new_rows)
            cat_df = cat_df.reindex(columns=target_columns)
            
            save_csv_name = f"{cat_name}.csv"
            save_path = os.path.join(output_root, save_csv_name)
            cat_df.to_csv(save_path, index=False)
            print(f"📄 已生成表格: {save_path}")

    print("\n所有处理完成！请检查 NIFD_Processed 文件夹。")

if __name__ == '__main__':
    # 检查 MRI 文件夹是否存在
    if not os.path.exists(source_image_dir):
        print(f"⚠️ 严重警告: 当前目录下找不到 '{source_image_dir}' 文件夹！")
        print(f"当前工作目录是: {os.getcwd()}")
        print("请确保你的 nii 文件都在当前目录下的 MRI 文件夹里。")
    else:
        process_nifd_dataset()

正在读取 NIFD.csv...

========== 正在处理类别: NC (COG=0) ==========
该类别在 CSV 中共有 136 条记录。
✅ 成功复制并处理了 136 个文件。
📄 已生成表格: ./NIFD_Processed\NC.csv

========== 正在处理类别: MCI (COG=1) ==========
该类别在 CSV 中共有 0 条记录。

========== 正在处理类别: AD (COG=2) ==========
该类别在 CSV 中共有 149 条记录。
✅ 成功复制并处理了 149 个文件。
📄 已生成表格: ./NIFD_Processed\AD.csv

所有处理完成！请检查 NIFD_Processed 文件夹。
